In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA quarantine;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS github_lakehouse.quarantine.quarantine_push_event (
    -- Common fields and original values
    event_id             STRING,
    event_type           STRING,
    event_timestamp      TIMESTAMP,
    event_timestamp_raw  STRING,
    event_date           DATE,
    actor_id             BIGINT,
    actor_id_raw         STRING,
    actor_login          STRING,
    repo_id              BIGINT,
    repo_id_raw          STRING,
    repo_name            STRING,

    -- Bronze lineage
    _source_file         STRING,
    _source_date         DATE,
    _ingested_at         TIMESTAMP,

    -- PushEvent fields and original values
    push_commit_count               BIGINT,
    push_commit_count_raw           STRING,
    push_distinct_commit_count      BIGINT,
    push_distinct_commit_count_raw  STRING,

        -- Quarantine evidence and metadata
    raw_json           STRING,
    _corrupt_record    STRING,
    quarantine_reason  STRING,
    quarantined_at     TIMESTAMP,

    -- Stable idempotency key: SHA-256 of (_source_file, _source_date, raw_json).
    -- Built in 01_push_events, cell 15.
    quarantine_key     STRING
)
USING DELTA;

In [0]:
# One-off, rerun-safe schema migration.
# CREATE TABLE IF NOT EXISTS never changes an existing table,
# so add quarantine_key only if it is missing.
QUARANTINE_TABLE = "github_lakehouse.quarantine.quarantine_push_event"

if "quarantine_key" in spark.table(QUARANTINE_TABLE).columns:
    print("quarantine_key already present; nothing to do.")
else:
    spark.sql(f"""
        ALTER TABLE {QUARANTINE_TABLE}
        ADD COLUMNS (
            quarantine_key STRING
            COMMENT 'SHA-256 of _source_file, _source_date, raw_json'
        )
    """)
    print("Added quarantine_key.")